In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from IPython.display import display

import matplotlib.pyplot as plt
REPO_DIR = os.environ['REPO_DIR']
DATA_DIR = os.environ['DATA_DIR']
sys.path.append(REPO_DIR)

from tqdm import tqdm

import os
import json
import pandas as pd
pd.set_option('display.max_colwidth', None)  # or use -1 for older pandas versions
from pathlib import Path
from collections import defaultdict
from utils.utils import read_segmentation_results, html_colors
from src.analysis import *

In [ ]:
run_type = "test"

subject_model_name = "google/gemma-3-4b-it"
explainer_model_name = "google/gemma-3-27b-it"

# subject_model_name = "OpenGVLab/InternVL3-14B"
# explainer_model_name = "OpenGVLab/InternVL3-14B"
layer = "mid"
overlay_type = "masks"

max_coeff_steering = 200
max_coeff_steered_topk = 50
results_experiment = load_full_results(subject_model_name, explainer_model_name, run_type,
                                        max_coeff_steering, max_coeff_steered_topk, overlay_type, layer, max_latents_test=1000)


In [ ]:
results_experiment['top_k'].keys()

In [ ]:
results_experiment['top_k']['gen_img_mean_auroc']

In [ ]:
results_experiment['steering']['mean_iou']

In [ ]:
results_list = []

for method, results in results_experiment.items():
    row = {
        "Method": method,
        "Mean IOU": round(results['mean_iou'], 3),
        "AUROC": round(results['gen_img_mean_auroc'], 3),
        "Mean Activation": round(results['gen_img_mean_mean'], 3),
        "Clip Score": round(results['clip_mean'], 3)
    }
    results_list.append(row)

results_df = pd.DataFrame(results_list)

# Display or use results_df as needed
display(results_df)
# os.makedirs(f"{OUTPUT}/paper_results", exist_ok=True)
# save_csv_path = f"{OUTPUT}/paper_results/{subject_model_name.replace('/', '_')}-{explainer_model_name.replace('/', '_')}-{layer}-{run_type}-{overlay_type}-evaluation_results.csv"
#results_df.to_csv(save_csv_path, index=False)


In [ ]:
from config import get_top_k_images_path

max_latents_test = 1000
# Default explainer method names
rand_vector_steering_explainer_method_name = f"steering_google_gemma-3-27b-it_sampling-False_blank_input__COEFF-200_DIRTYPE-random_norm_matched"
rand_perm_steering_explainer_method_name = f"steering_google_gemma-3-27b-it_sampling-False_blank_input__COEFF-200_DIRTYPE-permuted"

# Get path where the features explanation-related files are stored
outputs_path = get_outputs_path(subject_model_name, layer, run_type)
# results_experiment = {}
features_dir = get_top_k_images_path(subject_model_name, layer)
shared_kwargs = dict(features_dir=features_dir, run_type=run_type, max_latents_test=max_latents_test)
shared_kwargs

In [ ]:
results_experiment['rand_vector_steering'] = get_all_results(outputs_path, rand_vector_steering_explainer_method_name, **shared_kwargs)
results_experiment['rand_perm_steering'] = get_all_results(outputs_path, rand_perm_steering_explainer_method_name, **shared_kwargs)

In [ ]:
results_list = []

for method, results in results_experiment.items():
    row = {
        "Method": method,
        "Mean IOU": round(results['mean_iou'], 3),
        "AUROC": round(results['gen_img_mean_auroc'], 3),
        "Mean Activation": round(results['gen_img_mean_mean'], 3),
        "Clip Score": round(results['clip_mean'], 3)
    }
    results_list.append(row)

results_df = pd.DataFrame(results_list)

# Display or use results_df as needed
display(results_df)
os.makedirs(f"paper_results", exist_ok=True)
save_csv_path = f"paper_results/{subject_model_name.replace('/', '_')}-{explainer_model_name.replace('/', '_')}-{layer}-{run_type}-{overlay_type}-evaluation_results.csv"
results_df.to_csv(save_csv_path, index=False)


In [ ]:
df_dict = {}
for explanation_type in results_experiment.keys():
    df_dict[explanation_type], latent_idx_to_id = group_results_in_df(results_experiment[explanation_type])
merged_df = merge_all_dfs(df_dict, method_a='top_k', method_b='steered_top_k', method_c='steering')

In [ ]:
for metric in ["iou_score", "clip_score", "synth_act_score"]:#, "synth_act_score"]:
    counter_method_dict = perform_pairwise_tests(merged_df, metric)
    print(counter_method_dict)